In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [27]:
cleaned_df = pd.read_csv("cleaned_youtube_data.csv")
cleaned_df.head()

,video_id,date,views,likes,comments,watch_time_minutes,video_length_minutes,subscribers,ad_revenue_usd,day,...,device_Mobile,device_TV,device_Tablet,country_AU,country_CA,country_DE,country_IN,country_UK,country_US,engagement_rate
0,vid_3092,2024-09-24 10:50:40.993199,9936,1221,320,26497.21,2.86,228086,203.178237,24,...,0,1,0,0,0,0,1,0,0,0.1551
1,vid_3459,2024-09-22 10:50:40.993199,10017,642,346,15209.75,23.74,736015,140.880508,22,...,0,0,1,0,1,0,0,0,0,0.0986
2,vid_4784,2024-11-21 10:50:40.993199,10097,1979,187,57332.66,26.20,240534,360.134008,21,...,0,1,0,0,1,0,0,0,0,0.2145
3,vid_4078,2025-01-28 10:50:40.993199,10034,1191,242,31334.52,11.77,434482,224.638261,28,...,1,0,0,0,0,0,0,1,0,0.1428
4,vid_3522,2025-04-28 10:50:40.993199,9889,1858,477,15665.67,6.64,42030,165.514388,28,...,1,0,0,0,1,0,0,0,0,0.2361


In [25]:
cleaned_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 33 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   video_id                120000 non-null  str    
 1   date                    120000 non-null  str    
 2   views                   120000 non-null  int64  
 3   likes                   120000 non-null  int64  
 4   comments                120000 non-null  int64  
 5   watch_time_minutes      120000 non-null  float64
 6   video_length_minutes    120000 non-null  float64
 7   subscribers             120000 non-null  int64  
 8   ad_revenue_usd          120000 non-null  float64
 9   day                     120000 non-null  int64  
 10  month                   120000 non-null  int64  
 11  year                    120000 non-null  int64  
 12  day_of_week             120000 non-null  int64  
 13  is_weekend              120000 non-null  int64  
 14  retention_rate          120000 

In [28]:
drop_cols = [
    'video_id', 'date', 'day', 'month', 'year', 'day_of_week', 'is_weekend',
    'category_Education', 'category_Entertainment', 'category_Gaming',
    'category_Lifestyle', 'category_Music', 'category_Tech',
    'device_Desktop', 'device_Mobile', 'device_TV', 'device_Tablet',
    'country_AU', 'country_CA', 'country_DE', 'country_IN', 'country_UK', 'country_US',
    'subscribers'  # using log instead
]

df = cleaned_df.drop(columns=drop_cols)

df.columns

Index(['views', 'likes', 'comments', 'watch_time_minutes',
       'video_length_minutes', 'ad_revenue_usd', 'retention_rate',
       'log_subscribers', 'engagement_rate'],
      dtype='str')

In [29]:
from sklearn.model_selection import train_test_split

X = df.drop('ad_revenue_usd', axis=1)
y = df['ad_revenue_usd']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test, show_table=True):
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    print(f"\n{name}")
    print("-" * 30)
    print("Train R2:", r2_score(y_train, train_pred))
    print("Test R2 :", r2_score(y_test, test_pred))
    print("MAE     :", mean_absolute_error(y_test, test_pred))
    print("RMSE    :", np.sqrt(mean_squared_error(y_test, test_pred))

    )

    # 🔥 Comparison Table
    comparison = pd.DataFrame({
        'Actual': y_test.values,
        'Predicted': test_pred
    })

    comparison['Error'] = comparison['Actual'] - comparison['Predicted']
    comparison['Absolute_Error'] = abs(comparison['Error'])

    if show_table:
        print("\nSample Predictions (Top 10):")
        print(comparison.head())

        print("\nTop 10 Largest Errors:")
        print(comparison.sort_values(by='Absolute_Error', ascending=False).head())

    return comparison

In [31]:
#Linear Regression

evaluate_model(
    "Linear Regression",
    LinearRegression(),
    X_train, X_test, y_train, y_test
)


Linear Regression
------------------------------
Train R2: 0.9497227714523245
Test R2 : 0.9524804991697831
MAE     : 3.097473118806488
RMSE    : 13.49332393172143

Sample Predictions (Top 10):
       Actual   Predicted     Error  Absolute_Error
0  352.853521  352.816728  0.036793        0.036793
1  341.983788  342.091952 -0.108164        0.108164
2  204.586883  204.439932  0.146951        0.146951
3  176.835670  176.453416  0.382255        0.382255
4  270.842839  270.917394 -0.074555        0.074555

Top 10 Largest Errors:
           Actual   Predicted       Error  Absolute_Error
22216  138.085409  253.386271 -115.300861      115.300861
5870   138.682961  252.338759 -113.655799      113.655799
22981  147.800675  257.118943 -109.318269      109.318269
9379   148.225106  256.987552 -108.762446      108.762446
6294   369.099749  260.975211  108.124538      108.124538


,Actual,Predicted,Error,Absolute_Error
0,352.853521,352.816728,0.036793,0.036793
1,341.983788,342.091952,-0.108164,0.108164
2,204.586883,204.439932,0.146951,0.146951
3,176.835670,176.453416,0.382255,0.382255
4,270.842839,270.917394,-0.074555,0.074555
...,...,...,...,...
23995,322.670739,322.272805,0.397934,0.397934
23996,221.480597,221.558733,-0.078136,0.078136
23997,171.931110,172.078301,-0.147191,0.147191
23998,261.959118,261.792506,0.166611,0.166611


In [32]:
#Ridge Regression

evaluate_model(
    "Ridge Regression",
    Ridge(),
    X_train, X_test, y_train, y_test
)


Ridge Regression
------------------------------
Train R2: 0.9497218681718187
Test R2 : 0.9524861164384698
MAE     : 3.094191714773485
RMSE    : 13.492526386873994

Sample Predictions (Top 10):
       Actual   Predicted     Error  Absolute_Error
0  352.853521  352.812301  0.041220        0.041220
1  341.983788  342.054089 -0.070300        0.070300
2  204.586883  204.397125  0.189759        0.189759
3  176.835670  176.524819  0.310851        0.310851
4  270.842839  270.950943 -0.108104        0.108104

Top 10 Largest Errors:
           Actual   Predicted       Error  Absolute_Error
22216  138.085409  253.387710 -115.302301      115.302301
5870   138.682961  252.335928 -113.652968      113.652968
22981  147.800675  257.116038 -109.315363      109.315363
9379   148.225106  256.989777 -108.764672      108.764672
6294   369.099749  260.888306  108.211443      108.211443


,Actual,Predicted,Error,Absolute_Error
0,352.853521,352.812301,0.041220,0.041220
1,341.983788,342.054089,-0.070300,0.070300
2,204.586883,204.397125,0.189759,0.189759
3,176.835670,176.524819,0.310851,0.310851
4,270.842839,270.950943,-0.108104,0.108104
...,...,...,...,...
23995,322.670739,322.408821,0.261918,0.261918
23996,221.480597,221.542811,-0.062214,0.062214
23997,171.931110,171.975063,-0.043954,0.043954
23998,261.959118,261.806476,0.152642,0.152642


In [33]:
#Lasso Regression

evaluate_model(
    "Lasso Regression",
    Lasso(alpha=0.1),
    X_train, X_test, y_train, y_test
)


Lasso Regression
------------------------------
Train R2: 0.9497205002952762
Test R2 : 0.9524855433784268
MAE     : 3.0829405230624833
RMSE    : 13.492607752612361

Sample Predictions (Top 10):
       Actual   Predicted     Error  Absolute_Error
0  352.853521  352.852406  0.001115        0.001115
1  341.983788  342.047129 -0.063340        0.063340
2  204.586883  204.481343  0.105540        0.105540
3  176.835670  176.564490  0.271180        0.271180
4  270.842839  270.938214 -0.095375        0.095375

Top 10 Largest Errors:
           Actual   Predicted       Error  Absolute_Error
22216  138.085409  253.452321 -115.366911      115.366911
5870   138.682961  252.307297 -113.624337      113.624337
22981  147.800675  257.097243 -109.296568      109.296568
9379   148.225106  256.940085 -108.714979      108.714979
6294   369.099749  260.931312  108.168437      108.168437


,Actual,Predicted,Error,Absolute_Error
0,352.853521,352.852406,0.001115,0.001115
1,341.983788,342.047129,-0.063340,0.063340
2,204.586883,204.481343,0.105540,0.105540
3,176.835670,176.564490,0.271180,0.271180
4,270.842839,270.938214,-0.095375,0.095375
...,...,...,...,...
23995,322.670739,322.294499,0.376240,0.376240
23996,221.480597,221.558827,-0.078230,0.078230
23997,171.931110,171.965475,-0.034366,0.034366
23998,261.959118,261.770221,0.188896,0.188896


In [34]:
#Decision Tree

from sklearn.tree import DecisionTreeRegressor

evaluate_model(
    "Decision Tree",
    DecisionTreeRegressor(random_state=42),
    X_train, X_test, y_train, y_test
)


Decision Tree
------------------------------
Train R2: 1.0
Test R2 : 0.8980436485702823
MAE     : 5.724535297020605
RMSE    : 19.76469670403362

Sample Predictions (Top 10):
       Actual   Predicted     Error  Absolute_Error
0  352.853521  352.500390  0.353131        0.353131
1  341.983788  343.117076 -1.133288        1.133288
2  204.586883  204.103170  0.483713        0.483713
3  176.835670  176.859885 -0.024215        0.024215
4  270.842839  270.915568 -0.072729        0.072729

Top 10 Largest Errors:
           Actual   Predicted       Error  Absolute_Error
4843   146.433089  357.604989 -211.171900      211.171900
14919  343.653889  138.541165  205.112725      205.112725
13859  164.962209  361.513826 -196.551618      196.551618
165    348.502456  152.003326  196.499130      196.499130
9920   156.133392  351.853177 -195.719785      195.719785


,Actual,Predicted,Error,Absolute_Error
0,352.853521,352.500390,0.353131,0.353131
1,341.983788,343.117076,-1.133288,1.133288
2,204.586883,204.103170,0.483713,0.483713
3,176.835670,176.859885,-0.024215,0.024215
4,270.842839,270.915568,-0.072729,0.072729
...,...,...,...,...
23995,322.670739,323.516304,-0.845565,0.845565
23996,221.480597,221.557365,-0.076768,0.076768
23997,171.931110,171.093984,0.837125,0.837125
23998,261.959118,261.277446,0.681671,0.681671


In [35]:
#Random Forest
 
from sklearn.ensemble import RandomForestRegressor

evaluate_model(
    "Random Forest",
    RandomForestRegressor(
        n_estimators=50,     # reduce trees (faster)
        max_depth=15,        # prevent deep trees
        n_jobs=-1,           # use all CPU cores 🚀
        random_state=42
    ),
    X_train, X_test, y_train, y_test
)


Random Forest
------------------------------
Train R2: 0.9680280628026976
Test R2 : 0.9512884978690161
MAE     : 3.6189913505418434
RMSE    : 13.661512148852628

Sample Predictions (Top 10):
       Actual   Predicted     Error  Absolute_Error
0  352.853521  352.879114 -0.025593        0.025593
1  341.983788  342.122532 -0.138744        0.138744
2  204.586883  204.446884  0.139999        0.139999
3  176.835670  176.728783  0.106887        0.106887
4  270.842839  270.632506  0.210333        0.210333

Top 10 Largest Errors:
           Actual   Predicted       Error  Absolute_Error
6294   369.099749  245.652795  123.446954      123.446954
19382  151.653608  270.675468 -119.021859      119.021859
20653  134.808947  252.556298 -117.747351      117.747351
5788   174.769490  291.659482 -116.889991      116.889991
5870   138.682961  253.898652 -115.215691      115.215691


,Actual,Predicted,Error,Absolute_Error
0,352.853521,352.879114,-0.025593,0.025593
1,341.983788,342.122532,-0.138744,0.138744
2,204.586883,204.446884,0.139999,0.139999
3,176.835670,176.728783,0.106887,0.106887
4,270.842839,270.632506,0.210333,0.210333
...,...,...,...,...
23995,322.670739,323.696030,-1.025291,1.025291
23996,221.480597,221.984170,-0.503573,0.503573
23997,171.931110,170.932219,0.998890,0.998890
23998,261.959118,264.337367,-2.378249,2.378249


In [36]:
from sklearn.ensemble import GradientBoostingRegressor

evaluate_model(
    "Gradient Boosting",
    GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),
    X_train, X_test, y_train, y_test
)


Gradient Boosting
------------------------------
Train R2: 0.9500417502680254
Test R2 : 0.9521333072868101
MAE     : 3.6473840730226232
RMSE    : 13.542527379651712

Sample Predictions (Top 10):
       Actual   Predicted     Error  Absolute_Error
0  352.853521  353.004036 -0.150516        0.150516
1  341.983788  341.820863  0.162926        0.162926
2  204.586883  203.943237  0.643646        0.643646
3  176.835670  177.418110 -0.582439        0.582439
4  270.842839  271.513333 -0.670494        0.670494

Top 10 Largest Errors:
           Actual   Predicted       Error  Absolute_Error
22216  138.085409  253.350521 -115.265111      115.265111
5870   138.682961  251.903884 -113.220923      113.220923
6294   369.099749  257.690655  111.409095      111.409095
13965  363.479923  254.039367  109.440556      109.440556
22981  147.800675  256.844364 -109.043689      109.043689


,Actual,Predicted,Error,Absolute_Error
0,352.853521,353.004036,-0.150516,0.150516
1,341.983788,341.820863,0.162926,0.162926
2,204.586883,203.943237,0.643646,0.643646
3,176.835670,177.418110,-0.582439,0.582439
4,270.842839,271.513333,-0.670494,0.670494
...,...,...,...,...
23995,322.670739,321.840984,0.829755,0.829755
23996,221.480597,221.975363,-0.494766,0.494766
23997,171.931110,171.752647,0.178463,0.178463
23998,261.959118,259.982975,1.976142,1.976142


In [37]:
import pickle
from sklearn.linear_model import Lasso

final_model = Lasso(alpha=0.001)
final_model.fit(X_train, y_train)

pickle.dump(final_model, open("model.pkl", "wb"))
pickle.dump(X.columns, open("columns.pkl", "wb"))